# 数据探查：互动易问答 (isqa)

数据：`ISQA_SOURCE`（默认仓库 `data/field_research/isqa_20240101_20241231.jsonl`）

目标：彻底理解这份 **互动易 / 投资者问答(IR Q&A)** 数据的数据结构。

> 前 3 个逻辑 cell：`read_json` 加载 `df` → `df.head()` → 打印 `shape` 与 `columns`。


In [ ]:
import os
from pathlib import Path
import pandas as pd

ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
SOURCE = Path(os.environ.get("ISQA_SOURCE", ROOT / "data" / "field_research" / "isqa_20240101_20241231.jsonl"))
df = pd.read_json(SOURCE, lines=True)


In [ ]:
df.head()


In [ ]:
print("shape:", df.shape)
print("columns:")
print(df.columns.tolist())


In [ ]:
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["font.sans-serif"] = ["WenQuanYi Micro Hei", "WenQuanYi Zen Hei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

print("DataFrame shape:", df.shape)
print("内存占用(deep): {:.1f} MB".format(df.memory_usage(deep=True).sum() / 1024**2))
time_cols   = ["S_ASKDATE", "S_ANSWERDATE"]
text_cols   = ["S_QUESTIONCONTENT", "S_ANSWERCONTENT"]
cat_cols    = ["S_QUESTIONTYPE", "S_QUESTIONSOURCETYPE"]
entity_cols = ["S_INFO_WINDCODE", "QUESTION_PERSON", "ANSWER_PERSON"]
url_cols    = ["ORIGINALWEBSITE"]
id_cols     = ["EVENT_ID"]

field_meaning = {
    "S_INFO_WINDCODE": "股票 Wind 代码",
    "S_ASKDATE": "提问日期",
    "S_ANSWERDATE": "回答日期",
    "EVENT_ID": "问答事件ID",
    "S_QUESTIONTYPE": "问题类型(编码)",
    "S_QUESTIONCONTENT": "问题正文",
    "S_ANSWERCONTENT": "回答正文",
    "S_QUESTIONSOURCETYPE": "问题来源类型(编码)",
    "QUESTION_PERSON": "提问人",
    "ANSWER_PERSON": "回答人",
    "ORIGINALWEBSITE": "原始链接",
}
pd.DataFrame({"字段含义": field_meaning}).reindex(df.columns)


## 1. 字段类型 / 缺失 / 唯一值总览


In [ ]:
# ============================================================
# 1. 字段类型 / 缺失 / 唯一值总览
# ============================================================
overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "null": df.isna().sum(),
    "null_pct": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(dropna=False),
})
overview.sort_values(["null_pct", "n_unique"], ascending=[False, False])


## 2. 主键 / 唯一性 / 重复


In [ ]:
# ============================================================
# 2. 主键 / 唯一性 / 重复
# ============================================================
print("总行数:", len(df))
print("完全重复的行数 =", int(df.duplicated().sum()))
print("去掉 ID 列(如有)后仍完全重复的记录数 =", int(df.drop(columns=[c for c in ["EVENT_ID"] if c in df.columns]).duplicated().sum()))
print("EVENT_ID 非空 = %d，唯一值 = %d" % (
    int(df["EVENT_ID"].notna().sum()), df["EVENT_ID"].nunique()))

print("\n股票(Wind代码)唯一数 =", df["S_INFO_WINDCODE"].nunique())
print("Top20 被提问股票：")
print(df["S_INFO_WINDCODE"].value_counts().head(20).to_string())


## 3. 时间维度（提问 / 回答 日期）


In [ ]:
# ============================================================
# 3. 时间维度（提问 / 回答 日期）
# ============================================================
ask = pd.to_datetime(df["S_ASKDATE"], format="%Y%m%d", errors="coerce")
ans = pd.to_datetime(df["S_ANSWERDATE"], format="%Y%m%d", errors="coerce")
print("S_ASKDATE  解析失败:", int(ask.isna().sum()), "| 范围:", ask.min(), "→", ask.max())
print("S_ANSWERDATE 为空(未/暂无回答)行数:", int(ans.isna().sum()))
print("回答日期范围:", ans.min(), "→", ans.max())

lag_days = (ans - ask).dt.days
print("\n回答滞后天数(答-问): min/中位/均值/max = %.0f / %.0f / %.1f / %.0f" % (
    lag_days.min(), lag_days.median(), lag_days.mean(), lag_days.max()))
print(lag_days.describe())

monthly = ask.dt.to_period("M").value_counts().sort_index()
print("\n按月提问量：")
print(monthly)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
monthly.plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("提问量按月分布")
axes[0].tick_params(axis="x", rotation=60)
axes[1].hist(lag_days.dropna(), bins=50, color="coral")
axes[1].set_title("回答滞后天数分布")
plt.tight_layout()
plt.show()


## 4. 文本字段（问题 / 回答 正文）


In [ ]:
# ============================================================
# 4. 文本字段（问题 / 回答 正文）
# ============================================================
qlen = df["S_QUESTIONCONTENT"].str.len()
print("S_QUESTIONCONTENT 无缺失 =", bool(df["S_QUESTIONCONTENT"].notna().all()),
      "| 长度 min/中位/均值/max = %d / %.1f / %.1f / %d" % (
          qlen.min(), qlen.median(), qlen.mean(), qlen.max()))

ans_content = df["S_ANSWERCONTENT"].dropna()
print("\nS_ANSWERCONTENT 非空 %d / %d，缺失 %d (%.2f%%)" % (
    len(ans_content), len(df),
    int(df["S_ANSWERCONTENT"].isna().sum()), df["S_ANSWERCONTENT"].isna().mean() * 100))
alen = ans_content.str.len()
print("S_ANSWERCONTENT(非空) 长度：min/中位/均值/max = %d / %.1f / %.1f / %d" % (
    alen.min(), alen.median(), alen.mean(), alen.max()))

print("\n问题去重: 问题唯一值 %d / 行数 %d，重复问题行数 = %d" % (
    df["S_QUESTIONCONTENT"].nunique(), len(df), int(df["S_QUESTIONCONTENT"].duplicated().sum())))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(qlen, bins=50, color="steelblue")
axes[0].set_title("问题长度分布")
axes[1].hist(alen, bins=50, color="coral")
axes[1].set_title("回答长度分布")
plt.tight_layout()
plt.show()

print("\n--- 最长回答样本 ---")
i = alen.idxmax()
print("index=%d, 股票=%s, 问题=%s" % (i, df.loc[i, "S_INFO_WINDCODE"], df.loc[i, "S_QUESTIONCONTENT"][:80]))
print(df.loc[i, "S_ANSWERCONTENT"][:1500])
print("\n--- 最短(非空)回答样本 ---")
j = alen.idxmin()
print("index=%d, 股票=%s" % (j, df.loc[j, "S_INFO_WINDCODE"]))
print(repr(df.loc[j, "S_ANSWERCONTENT"]))


## 5. 分类字段（问题类型 / 来源类型 编码）


In [ ]:
# ============================================================
# 5. 分类字段（问题类型 / 来源类型 编码）
# ============================================================
for c in cat_cols:
    print("=" * 70)
    print("%s  唯一值数=%d" % (c, df[c].nunique(dropna=True)))
    print(df[c].value_counts(dropna=False).head(20).to_string())
    print()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for i, c in enumerate(cat_cols):
    df[c].value_counts(dropna=False).head(15).plot(kind="bar", ax=axes[i], color="steelblue")
    axes[i].set_title(c)
    axes[i].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


## 6. 提问人 / 回答人 分布


In [ ]:
# ============================================================
# 6. 提问人 / 回答人 分布
# ============================================================
print("提问人去重人数 =", df["QUESTION_PERSON"].nunique())
print("提问人 Top20：")
print(df["QUESTION_PERSON"].value_counts().head(20).to_string())

print("\n回答人去重数 =", df["ANSWER_PERSON"].nunique())
print("回答人 Top20：")
print(df["ANSWER_PERSON"].value_counts(dropna=False).head(20).to_string())


## 7. 原始链接


In [ ]:
# ============================================================
# 7. 原始链接
# ============================================================
dom = df["ORIGINALWEBSITE"].dropna().str.extract(r"https?://([^/]+)", expand=False)
print("ORIGINALWEBSITE 域名 Top20：")
print(dom.value_counts(dropna=False).head(20).to_string())


## 8. 字段交叉关系


In [ ]:
# ============================================================
# 8. 字段交叉关系
# ============================================================
tmp = df.copy()
tmp["HAS_ANSWER"] = tmp["S_ANSWERCONTENT"].notna()
print("问题类型(S_QUESTIONTYPE) × 是否有回答：")
print(pd.crosstab(tmp["S_QUESTIONTYPE"], tmp["HAS_ANSWER"], margins=True))

print("\n问题来源类型(S_QUESTIONSOURCETYPE) × 是否有回答：")
print(pd.crosstab(tmp["S_QUESTIONSOURCETYPE"], tmp["HAS_ANSWER"], margins=True))

print("\n回答人(是否为空) 与 问题来源类型：")
print(pd.crosstab(tmp["S_QUESTIONSOURCETYPE"], tmp["ANSWER_PERSON"].notna(), margins=True))


## 9. 原始数据抽样查看


In [ ]:
# ============================================================
# 9. 原始数据抽样查看
# ============================================================
sample = df.sample(n=3, random_state=42)
for i, (idx, row) in enumerate(sample.iterrows(), 1):
    print("=" * 80)
    print("样本 %d / 3   (原始 index=%d)" % (i, idx))
    print("-" * 80)
    for col in df.columns:
        val = row[col]
        if isinstance(val, str) and len(val) > 300:
            val = val[:300] + "  ......[截断]"
        print("%-22s : %s" % (col, val))
    print()


## 10. 结论摘要（自动汇总）


In [ ]:
# ============================================================
# 10. 结论摘要（自动汇总）
# ============================================================
ask_dt = pd.to_datetime(df["S_ASKDATE"], format="%Y%m%d", errors="coerce")
ans_dt = pd.to_datetime(df["S_ANSWERDATE"], format="%Y%m%d", errors="coerce")
lines = []
lines.append("文件: %s" % SOURCE)
lines.append("总记录数: %d 行 × %d 列" % df.shape)
lines.append("提问日期范围: %s ~ %s" % (ask_dt.min(), ask_dt.max()))
lines.append("回答日期范围: %s ~ %s" % (ans_dt.min(), ans_dt.max()))
lines.append("完全重复行: %d" % int(df.duplicated().sum()))
lines.append("股票(Wind代码) %d 只；提问人 %d；回答人 %d" % (
    df["S_INFO_WINDCODE"].nunique(), df["QUESTION_PERSON"].nunique(),
    df["ANSWER_PERSON"].nunique()))
lines.append("回答正文缺失率: %.2f%%" % (df["S_ANSWERCONTENT"].isna().mean()*100))
lines.append("问题类型取值: %s" % sorted(df["S_QUESTIONTYPE"].dropna().unique().tolist()))
lines.append("问题来源类型取值: %s" % sorted(df["S_QUESTIONSOURCETYPE"].dropna().unique().tolist()))
lines.append("回答滞后(答-问) 中位天数: %.0f" % (ans_dt - ask_dt).dt.days.median())
print("\n".join(lines))
